In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import sys
sys.path.insert(0, '/content/drive/My Drive/Colab Notebooks/nirdizati-light_Test')

In [ ]:
#!pip install pm4py
#!pip install holidays
#!pip install dateparser

In [ ]:
# Install nirdizati-light package
!pip install git+https://github.com/rgraziosi-fbk/nirdizati-light

# If asked to reload runtime to update numpy version, click "yes"

  Cloning https://github.com/rgraziosi-fbk/nirdizati-light to /tmp/pip-req-build-pxptweir
  Running command git clone --filter=blob:none --quiet https://github.com/rgraziosi-fbk/nirdizati-light /tmp/pip-req-build-pxptweir
  Resolved https://github.com/rgraziosi-fbk/nirdizati-light to commit b6dbd5c6f730e1170212a13a602d8071cb4705a1
  Preparing metadata (setup.py) ... done
  Cloning https://github.com/abuliga/DiCE.git (to revision origin/main) to /tmp/pip-install-qyeg31hu/dice-ml_5a5be450ec9f4464806d274c21aad8ad
  Running command git clone --filter=blob:none --quiet https://github.com/abuliga/DiCE.git /tmp/pip-install-qyeg31hu/dice-ml_5a5be450ec9f4464806d274c21aad8ad
  Running command git checkout -b origin/main --track origin/origin/main
  Switched to a new branch 'origin/main'
  Branch 'origin/main' set up to track remote branch 'origin/main' from 'origin'.
  Resolved https://github.com/abuliga/DiCE.git to commit 386bdb9fd431b962f791649707bc836a095656a3
  Preparing metadata (setup.py) 

In [ ]:
# Reinstall cupy-cuda12 to ensure compatibility
!pip uninstall -y cupy-cuda12x
!pip install cupy-cuda12x|
#!pip install -U cupy-cuda12x
#!pip install cudf-cu12 --extra-index-url=https://pypi.nvidia.com
#import os
#os.environ["LD_LIBRARY_PATH"] = "/usr/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

/bin/bash: -c: line 2: syntax error: unexpected end of file


In [ ]:
# Download an example log
!mkdir datasets
!wget "https://drive.google.com/uc?export=download&id=1qcx8F7nFo20kENuvBKWfQLidgi54adlv" -O datasets/bpic2012_O_ACCEPTED-COMPLETE_trunc.xes


mkdir: cannot create directory ‘datasets’: File exists
--2025-11-15 18:45:37--  https://drive.google.com/uc?export=download&id=1qcx8F7nFo20kENuvBKWfQLidgi54adlv
Resolving drive.google.com (drive.google.com)... 173.194.216.102, 173.194.216.100, 173.194.216.113, ...
Connecting to drive.google.com (drive.google.com)|173.194.216.102|:443... connected.
HTTP request sent, awaiting response... 303 See Other
Location: https://drive.usercontent.google.com/download?id=1qcx8F7nFo20kENuvBKWfQLidgi54adlv&export=download [following]
--2025-11-15 18:45:37--  https://drive.usercontent.google.com/download?id=1qcx8F7nFo20kENuvBKWfQLidgi54adlv&export=download
Resolving drive.usercontent.google.com (drive.usercontent.google.com)... 74.125.26.132, 2607:f8b0:400c:c04::84
Connecting to drive.usercontent.google.com (drive.usercontent.google.com)|74.125.26.132|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 99833831 (95M) [application/octet-stream]
Saving to: ‘datasets/bpic2012_O_ACCE

# Prepare environment

In [ ]:
import os
import random
import numpy as np
print("My numpy version is: ", np.__version__)

import pandas as pd

import numba
print("My numba version is: ", numba.__version__)
from numba import cuda
print(cuda.gpus)

import pm4py


CudaSupportError: Error at driver init: 

CUDA driver library cannot be found.
If you are sure that a CUDA driver is installed,
try setting environment variable NUMBA_CUDA_DRIVER
with the file path of the CUDA driver shared library.
:

In [ ]:
from nirdizati_light.log.common import get_log, split_train_val_test
from nirdizati_light.encoding.common import get_encoded_df, EncodingType
from nirdizati_light.encoding.constants import TaskGenerationType, PrefixLengthStrategy, EncodingTypeAttribute
from nirdizati_light.encoding.time_encoding import TimeEncodingType
from nirdizati_light.labeling.common import LabelTypes

# Encoding

# Explanation of the `get_encoded_df` Function Parameters

The function `get_encoded_df` encodes data from a process log to create:
1. An **encoder object** (`encoder`) that processes features or labels.
2. A fully encoded **DataFrame** (`full_df`) for machine learning tasks.

## Parameters:

1. **`log`**:
   - Likely a process event log or dataset containing sequential events for one or more processes.
   - Columns might include `case_id`, `activity`, `timestamp`, and other attributes.

2. **`feature_encoding_type`**:
   - Specifies the type of feature encoding:
     - `frequency`: Frequency-based encoding (e.g., count of activities).
     - `ordinal`: Maps categorical features to integers.
     - `one_hot`: One-hot encoding for categorical features.
     - `embedding`: Use embeddings for numerical representation.

3. **`prefix_length`**:
   - Determines the length of process prefixes to be used.
   - Affects how much history (number of events) is considered for features.

4. **`prefix_length_strategy`**:
   - Defines how prefixes are handled:
     - `fixed`: Use a fixed number of steps (e.g., only first $N$ steps).
     - `dynamic`: Variable-length prefixes, potentially based on case characteristics.

5. **`time_encoding_type`**:
   - Encodes timestamp-related features (e.g., weekday, hour, month, time since last event, time since case start):
     - `NONE`: Does not encode the relative time attributes.
     - `DATE`: Extracts date-related features like day-of-week or month.
     - `DURATION`: Encodes time differences between events, time since case start, etc.
     - `DATE_AND_DURATION`: Combines date and duration encodings.

6. **`attribute_encoding`**:
   - Specifies how to encode additional attributes:
     - `LABEL`: Encode attributes as labels (e.g., 'A', 'B', 'C').
     - `ONEHOT`: Use one-hot encoding for attributes.

7. **`padding`**:
   - Ensures uniform input length by padding shorter prefixes with default values (e.g., zeros).
   - Useful for models like RNNs or LSTMs requiring uniform input dimensions.

8. **`labeling_type`**:
   - Defines how labels are generated for classification or prediction:
     - `NEXT_ACTIVITY`: Predict the next activity in the process.
     - `ATTRIBUTE_STRING`: Predict the final outcome of a process instance.
     - `ATTRIBUTE_NUMBER`: Predict a numerical attribute value (e.g., case duration).
     - `REMAINING_TIME`: Predict the remaining time until case completion.
     - `DURATION`: Predict the duration of a case.

9. **`task_generation_type`**:
   - Specifies the type of task:
     - `CLASSIFICATION`: Assign discrete classes (e.g., case outcome as 'successful' or 'failed').
     - `REGRESSION`: Predict continuous values (e.g., REMAINING_TIME, DURATION).

10. **`target_event`**:
    - Focuses on a specific event in the process for next activity predictive tasks.
    - Example: A particular milestone activity or timestamp.

## Outputs:

1. **`encoder`**:
   - Object or function that transforms raw features into encoded forms.
   - Stores mappings (e.g., one-hot encoding maps, label encoders).

2. **`full_df`**:
   - Fully encoded DataFrame for model training.
   - Includes encoded features, labels, and optional padding for consistency.


## <span style="color: red;">WARNING:</span> When using encoder.decode(full_df), this modifies the full_df DataFrame in place. To encode the dataframe again, you need to re-run the encoding process with encoder.encode(full_df).


Encoding feature categoriche: OneHot

In [ ]:
CONF = {
    # path to log
    'data': 'BPIC11_f1.csv',
    # train-validation-test set split percentages
    'train_val_test_split': [0.7, 0.1, 0.2],

    # path to output folder
    'output': 'output_data',

    'prefix_length_strategy': PrefixLengthStrategy.FIXED.value,
    'prefix_length': 15,

    # whether to use padding or not in encoding
    'padding': True,
    # which encoding to use
    'feature_selection': EncodingType.COMPLEX.value,
    # which attribute encoding to use
    # Forse vogliamo OneHot
    'attribute_encoding': EncodingTypeAttribute.ONEHOT.value,
    # which time encoding to use
    'time_encoding': TimeEncodingType.NONE.value,

    # the label to be predicted (e.g. outcome, next activity)
    'labeling_type': LabelTypes.ATTRIBUTE_STRING.value,
    # whether the model should be trained on the specified prefix length (ONLY_THIS) or to every prefix in range [1, prefix_length] (ALL_IN_ONE)
    'task_generation_type': TaskGenerationType.ALL_IN_ONE.value,
    'target_event': None,
    'seed': 49,
}

In [ ]:
log = get_log(filepath=CONF['data'], separator=';')

In [ ]:
log[0][0].keys()

In [ ]:
encoder, full_df = get_encoded_df(
  log=log,
  feature_encoding_type=CONF['feature_selection'],
  prefix_length=CONF['prefix_length'],
  prefix_length_strategy=CONF['prefix_length_strategy'],
  time_encoding_type=CONF['time_encoding'],
  attribute_encoding=CONF['attribute_encoding'],
  padding=CONF['padding'],
  labeling_type=CONF['labeling_type'],
  task_generation_type=CONF['task_generation_type'],
  target_event=CONF['target_event'],
)

In [ ]:
#encoder.decode(full_df)
#encoder.get_values(full_df)

In [ ]:
label_columns = [item for item in full_df.columns if 'label' in item ]

In [ ]:
label_columns

In [ ]:
full_df

In [ ]:
(full_df['trace_id'].unique() == full_df['trace_id'])

In [ ]:
from pandas import DataFrame
from funcy import flatten

In [ ]:
import math

In [ ]:
def get_tensor(df: DataFrame, prefix_length):
    # Prendo dalle colonne del dataframe gli attributi delle tracce a meno che non contengano 'prefix_"
    trace_attributes = [att for att in df.columns if 'prefix_' not in att]
    event_attributes = [att[:-2] for att in df.columns if att[-2:] == '_1']

    reshaped_data = {
            trace_index: {
                prefix_index:
                    list(flatten(
                        feat_values if isinstance(feat_values, tuple) else [feat_values]
                        for feat_name, feat_values in trace.items()
                        if feat_name in trace_attributes + [event_attribute + '_' + str(prefix_index) for event_attribute in event_attributes]
                    ))
                for prefix_index in range(1, prefix_length + 1)
            }
            for trace_index, trace in df.iterrows()
    }

    flattened_features = max(
        len(reshaped_data[trace][prefix])
        for trace in reshaped_data
        for prefix in reshaped_data[trace]
    )

    tensor = np.zeros((
        len(df),                # sample
        prefix_length,          # time steps
        flattened_features      # features x single time step (trace and event attributes)
    ))

    for i, trace_index in enumerate(reshaped_data):  # prefix
        for j, prefix_index in enumerate(reshaped_data[trace_index]):  # steps of the prefix
            for single_flattened_value in range(len(reshaped_data[trace_index][prefix_index])):
                tensor[i, j, single_flattened_value] = reshaped_data[trace_index][prefix_index][single_flattened_value]

    return tensor

In [ ]:
def check_att(att, prefix_length):
     try:
          cur_length = int( att[ len(att) - att[::-1].index('_') : ] )
          return not ('prefix_' in att or 'Prefix_' in att) and cur_length <= 0
         #or not cur_length in range(1,prefix_length+1)
     except ValueError:
          return True

In [ ]:
def get_tensor_alt(df: DataFrame, prefix_length):
    # Prendo dalle colonne del dataframe gli attributi delle tracce a meno che non contengano 'prefix_"
#trace_attributes = [att for att in df.columns if 'prefix_' not in att]
    trace_attributes = [ att for att in df.columns if check_att(att, prefix_length) ]
    # Dagli attributi delle tracce prendo i nomi differenti togliendo il suffisso con l'indicazione numerale
    event_attributes = [att[:-2] for att in df.columns if att[-2:] == '_1' and not ('prefix_' in att or 'Prefix_' in att)]

    reshaped_data = {
            trace_index: {
                prefix_index:
                    list(flatten(
                        feat_values if isinstance(feat_values, tuple) else [feat_values]
                        for feat_name, feat_values in trace.items()
                        if feat_name in trace_attributes + [event_attribute + '_' + str(prefix_index) for event_attribute in event_attributes]
                    ))
                for prefix_index in range(1, prefix_length + 1)
            }
            for trace_index, trace in df.iterrows()
    }

    flattened_features = max(
        len(reshaped_data[trace][prefix])
        for trace in reshaped_data
        for prefix in reshaped_data[trace]
    )

    if (prefix_length==1):
        tensor = np.zeros((
            len(df),                # sample
            flattened_features      # features x single time step (trace and event attributes)
        ))
    else:
            tensor = np.zeros((
            len(df),                # sample
            prefix_length,          # time steps
            flattened_features      # features x single time step (trace and event attributes)
            ))

    for i, trace_index in enumerate(reshaped_data):  # prefix
        for j, prefix_index in enumerate(reshaped_data[trace_index]):  # steps of the prefix
            for single_flattened_value in range(len(reshaped_data[trace_index][prefix_index])):
                if (prefix_length==1):
                    tensor[i, single_flattened_value] = reshaped_data[trace_index][1][single_flattened_value]
                else:
                    tensor[i, j, single_flattened_value] = reshaped_data[trace_index][prefix_index][single_flattened_value]

    return tensor

## Analisi del funzionamento di get_tensor

In [ ]:
df = full_df.iloc[0:120]

In [ ]:
df

In [ ]:
# Funziona solo se prefix length è uguale a quello nel preprocess
prefix_length = 4

In [ ]:
trace_attributes = [ att for att in df.columns if check_att(att, prefix_length) ]

# ToDo

- [ ] trace_attributes vanno scartati dal tensore per training

In [ ]:
trace_attributes

In [ ]:
event_attributes = [att[:-2] for att in df.columns if att[-2:] == '_1']

In [ ]:
event_attributes = [att[:-2] for att in df.columns if att[-2:] == '_1' and not ('prefix_' in att or 'Prefix_' in att)]

In [ ]:
event_attributes

In [ ]:
reshaped_data = {
            trace_index: {
                prefix_index:
                    list(flatten(
                        feat_values if isinstance(feat_values, tuple) else [feat_values]
                        for feat_name, feat_values in trace.items()
                        if feat_name in trace_attributes + [event_attribute + '_' + str(prefix_index) for event_attribute in event_attributes]
                    ))
                for prefix_index in range(1, prefix_length + 1)
            }
            for trace_index, trace in df.iterrows()
    }

In [ ]:
def create_feature_list(trace, prefix_index):
    feature_list = []
    for feat_name, feat_values in trace.items():
        if feat_name in trace_attributes + [event_attribute + '_' + str(prefix_index) for event_attribute in event_attributes]:
            print(feat_name)
            if isinstance(feat_values, tuple):
               feature_list.append(feat_values)
            else:
               feature_list.append([feat_values])
    return list(flatten(feature_list))

In [ ]:
reshaped_data_alt = {
     		trace_index: {
			prefix_index:
    				create_feature_list(trace, prefix_index)
                	for prefix_index in range(1, prefix_length + 1)
		}
		for trace_index, trace in df.iterrows()
	}

In [ ]:
reshaped_data_alt

In [ ]:
prefix_index = 1
range1 = trace_attributes + [event_attribute + '_' + str(prefix_index) for event_attribute in event_attributes]
prefix_index = 2
range2 = trace_attributes + [event_attribute + '_' + str(prefix_index) for event_attribute in event_attributes]

In [ ]:
range1

In [ ]:
len(range2)

In [ ]:
row = next(df.iterrows())[1]

In [ ]:
row

In [ ]:
test_tensor = get_tensor_alt(df, 4)

In [ ]:
test_tensor

In [ ]:
T = test_tensor.sum(axis=1)

In [ ]:
T

In [ ]:
T[0,:]

## Funzione shape_label_df

In [ ]:
def shape_label_df(df: DataFrame):
    labels_list = df['label'].tolist()
    labels = np.zeros((len(labels_list), int(max(df['label'].nunique(), int(max(df['label'].values))) + 1)))
    for label_idx, label_val in enumerate(labels_list):
        labels[int(label_idx), int(label_val)] = 1

    return labels

In [ ]:
Lab = shape_label_df(full_df)

In [ ]:
Lab

In [ ]:
np.unique(Lab[:,2])

In [ ]:
Tens = get_tensor( full_df, 10 )

In [ ]:
Tens

In [ ]:
Tens[0,0,:]

In [ ]:
Tens[0,1,:]